In [1]:
# ...existing code...
# Test runner for extract_cophieu68 - paste into a notebook cell
import os, sys, time, json
from dataclasses import asdict, is_dataclass

# ensure project root on path
PROJECT_ROOT = "/mnt/c/Users/Admin/Downloads/Project/Github/ETL_Project"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# imports
from internal.dags.cophieu68_dag.extract.extract_cophieu68 import extract_cophieu68
from internal.dags.cophieu68_dag.extract.base_extract import Cophieu68BeautifulSoupCrawler
from internal.dags.ETL_Orchestra.main_orchestra_etl import ETLPipelineConfig

# init pipeline config and loggers
config = ETLPipelineConfig(
    config_path="/mnt/c/Users/Admin/Downloads/Project/Github/ETL_Project/internal/config/web_craw_config/cophieu68_config.yaml"
)

def safe_serialize(obj, _depth=0):
    try:
        if obj is None:
            return None
        if _depth > 3:
            return str(type(obj))
        if isinstance(obj, (list, tuple)):
            return [safe_serialize(x, _depth+1) for x in obj[:10]]  # limit length
        if is_dataclass(obj):
            return asdict(obj)
        # pandas DataFrame
        try:
            import pandas as pd
            if isinstance(obj, pd.DataFrame):
                return obj.head(3).to_dict(orient="records")
        except Exception:
            pass
        # fallback string
        return str(obj)[:1000]
    except Exception as e:
        return f"<serialize-error:{e}>"

# instantiate crawler correctly (base class expects pipeline_config, pipeline_logger)
# use __new__ + base initializer to avoid modifying extract_cophieu68 source
crawler = extract_cophieu68.__new__(extract_cophieu68)
Cophieu68BeautifulSoupCrawler.__init__(crawler, config.config, config.etl_extract_logger)
# ensure attributes extract_cophieu68.__init__ normally sets
try:
    crawler.endpoint = crawler.crawler_cfg.get("endpoints", {})
except Exception:
    crawler.endpoint = {}

summary = {"meta": {}, "results": {}}
summary["meta"]["base_urls"] = getattr(crawler, "urls", None)
summary["meta"]["endpoints"] = getattr(crawler, "endpoint", getattr(crawler, "crawler_cfg", {}).get("endpoints", None))

# determine symbols to test
symbols = []
try:
    cfg = getattr(crawler, "crawler_cfg", {}) or {}
    symbols = cfg.get("sample_symbols") or cfg.get("symbols") or []
except Exception:
    symbols = []

# fallback: try market list to fetch few symbols (may be slow)
if not symbols:
    try:
        symbols = crawler.crawl_market_list()[:5]
    except Exception:
        symbols = ["ACB", "VCB", "HPG"]

symbols = [s.upper() for s in symbols][:5]
summary["meta"]["test_symbols"] = symbols

# list of test calls per symbol
per_symbol_tests = [
    ("crawl_complete_stock_data", lambda c, s: c.crawl_complete_stock_data(s)),
    ("crawl_financial_ratios", lambda c, s: c.crawl_financial_ratios(s)),
    ("crawl_power_ratings", lambda c, s: c.crawl_power_ratings(s)),
    ("crawl_trading_data", lambda c, s: c.crawl_trading_data(s)),
    ("crawl_company_profile", lambda c, s: c.crawl_company_profile(s)),
    ("crawl_income_statement", lambda c, s: c.crawl_income_statement(s)),
    ("crawl_balance_sheet", lambda c, s: c.crawl_balance_sheet(s)),
    ("crawl_cashflow_statement", lambda c, s: c.crawl_cashflow_statement(s)),
]

# run per-symbol tests
for sym in symbols:
    summary["results"][sym] = {}
    for name, fn in per_symbol_tests:
        try:
            start = time.time()
            out = fn(crawler, sym)
            took = time.time() - start
            summary["results"][sym][name] = {
                "ok": True,
                "took_sec": round(took, 2),
                "summary": safe_serialize(out)
            }
            config.etl_extract_logger.info(f"Test {name} for {sym} OK ({took:.2f}s)")
        except Exception as e:
            summary["results"][sym][name] = {
                "ok": False,
                "error": str(e)
            }
            config.etl_extract_logger.error(f"Test {name} for {sym} ERROR: {e}")

# run non-symbol methods (market/industry list)
non_symbol_tests = [
    ("crawl_market_list_all", lambda c: c.crawl_market_list("all")),
    ("crawl_industry_list", lambda c: c.crawl_industry_list()),
    ("crawl_industry_info_sample", lambda c: c.crawl_industry_info(symbols[0]) if symbols else None),
]

for name, fn in non_symbol_tests:
    try:
        start = time.time()
        out = fn(crawler)
        took = time.time() - start
        summary["results"][name] = {"ok": True, "took_sec": round(took, 2), "summary": safe_serialize(out)}
        config.etl_extract_logger.info(f"Test {name} OK ({took:.2f}s)")
    except Exception as e:
        summary["results"][name] = {"ok": False, "error": str(e)}
        config.etl_extract_logger.error(f"Test {name} ERROR: {e}")

# optional: save summary
out_path = "/tmp/cophieu68_extract_test_summary.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Test complete. Summary saved to", out_path)
# display brief summary
for k, v in summary["results"].items():
    print(k, "->", "OK" if v.get("ok") else "ERR")
# # ...existing code...

Initializing ETLPipelineConfig... /mnt/c/Users/Admin/Downloads/Project/Github/ETL_Project/internal/config/web_craw_config/cophieu68_config.yaml


{"time":"2025-10-11T00:28:30", "level":"ERROR", "message":"Test crawl_complete_stock_data for ACB ERROR: string indices must be integers, not 'str'", "caller":"/tmp/ipykernel_20772/3768170541.py:106"}


{'name': 'cophieu68', 'url': 'https://cophieu68.vn', 'version': 1.0, 'service_name': 'data-api-go', 'description': 'ELT pipeline config: ingest raw HTML/JSON, persist to HDFS/MongoDB, transform with Spark, load to MySQL DW', 'logger': {'level': 'INFO', 'loki_enabled': False, 'loki_url': 'http://loki.monitoring.svc.cluster.local:3100/loki/api/v1/push', 'database_logger': {'storage_path': './logger_storage/cophieu68_elt_logger/database_logger', 'files': {'debug': 'database_debug.log', 'info': 'database_info.log', 'warning': 'database_warning.log', 'error': 'database_error.log'}, 'max_size_mb': 100, 'backup_count': 5}, 'etl_logger': {'extract_log': {'storage_path': './logger_storage/cophieu68_elt_logger/etl_logger/extract', 'files': {'debug': 'extract_debug.log', 'info': 'extract_info.log', 'warning': 'extract_warning.log', 'error': 'extract_error.log'}, 'max_size_mb': 100, 'backup_count': 5}, 'transform_log': {'storage_path': './logger_storage/cophieu68_elt_logger/etl_logger/transform', 

{"time":"2025-10-11T00:28:32", "level":"INFO", "message":"Test crawl_financial_ratios for ACB OK (1.38s)", "caller":"/tmp/ipykernel_20772/3768170541.py:100"}
{"time":"2025-10-11T00:28:32", "level":"ERROR", "message":"Test crawl_power_ratings for ACB ERROR: string indices must be integers, not 'str'", "caller":"/tmp/ipykernel_20772/3768170541.py:106"}
{"time":"2025-10-11T00:28:32", "level":"ERROR", "message":"Test crawl_trading_data for ACB ERROR: string indices must be integers, not 'str'", "caller":"/tmp/ipykernel_20772/3768170541.py:106"}
{"time":"2025-10-11T00:28:32", "level":"ERROR", "message":"Test crawl_company_profile for ACB ERROR: string indices must be integers, not 'str'", "caller":"/tmp/ipykernel_20772/3768170541.py:106"}
{"time":"2025-10-11T00:28:32", "level":"ERROR", "message":"Error fetching income report for ACB: Missing optional dependency 'lxml'.  Use pip or conda to install lxml.", "caller":"/mnt/c/Users/Admin/Downloads/Project/Github/ETL_Project/internal/dags/cophie

Test complete. Summary saved to /tmp/cophieu68_extract_test_summary.json
ACB -> ERR
VCB -> ERR
HPG -> ERR
crawl_market_list_all -> ERR
crawl_industry_list -> ERR
crawl_industry_info_sample -> ERR
